# 🎬 BackstageCommercials - Complete Video Processing

This notebook processes your video entirely on Google Colab:
1. Scene analysis
2. Product placement planning
3. FLUX model generation
4. Video rendering

**Result:** Video with naturally embedded product ads

---

## ⚙️ Setup

**Before running:**
1. Runtime → Change runtime type → **GPU (T4 or better)**
2. Upload your video file
3. Upload your product image

**Then:** Run all cells in order

---
## 📦 Step 1: Install Dependencies

In [ ]:
print("📦 Installing dependencies...\n")

# PyTorch with CUDA
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# Diffusion models
!pip install -q diffusers transformers accelerate safetensors

# Image/Video processing
!pip install -q opencv-python pillow numpy

# HuggingFace
!pip install -q huggingface-hub

# Quantization
!pip install -q optimum-quanto

# Finegrain toolbox for FLUX
!pip install -q git+https://github.com/finegrain-ai/finegrain-toolbox.git

# YOLO for segmentation
!pip install -q ultralytics

# Amazon Nova (for scene analysis - if API available)
!pip install -q openai python-dotenv

print("\n✅ All dependencies installed!")

---
## 🔍 Step 2: Verify GPU

In [ ]:
import torch

print("="*60)
print("GPU STATUS")
print("="*60)
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"Total Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"CUDA Version: {torch.version.cuda}")
    print("\n✅ GPU is ready!")
else:
    print("\n⚠️  WARNING: GPU not available!")
    print("Please change runtime: Runtime → Change runtime type → GPU")

print("="*60)

---
## 📤 Step 3: Upload Your Files

In [ ]:
from google.colab import files
import os

print("📤 Upload your files:")
print("1. Video file (e.g., movie.mp4)")
print("2. Product image (e.g., coffee.png with transparent background)")
print("\nClick 'Choose Files' button below...\n")

uploaded = files.upload()

print("\n✅ Files uploaded:")
for filename in uploaded.keys():
    size_mb = len(uploaded[filename]) / (1024 * 1024)
    print(f"   - {filename} ({size_mb:.2f} MB)")

# Auto-detect video and product files
video_file = None
product_file = None

for filename in uploaded.keys():
    if filename.endswith(('.mp4', '.avi', '.mov', '.mkv')):
        video_file = filename
    elif filename.endswith(('.png', '.jpg', '.jpeg')):
        product_file = filename

print(f"\n🎬 Video: {video_file}")
print(f"🖼️  Product: {product_file}")

---
## ⚙️ Step 4: Configuration

In [ ]:
# Configuration
PRODUCT_DESCRIPTION = "Coffee jar"  # ✏️ EDIT THIS: Describe your product

# Optional: Amazon Nova API keys (if you have them)
# If not provided, will use fallback methods
NOVA_API_KEY = ""  # Leave empty if you don't have it

# Processing options
USE_QUANTIZATION = True  # Set to True if you face memory issues
MIN_SHOT_SEC = 1.0       # Minimum scene duration
MAX_SHOT_SEC = 5.0       # Maximum scene duration

print("✅ Configuration set!")
print(f"Product: {PRODUCT_DESCRIPTION}")
print(f"Quantization: {USE_QUANTIZATION}")

---
## 🔍 Step 5: Scene Analysis

In [ ]:
import cv2
import numpy as np
import json

print("🔍 Analyzing video for best placement scenes...\n")

def find_scene_cuts(video_path, threshold=30.0):
    """Find scene cuts in video"""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    scene_cuts = [0]
    prev_frame = None
    
    for i in range(frame_count):
        ret, frame = cap.read()
        if not ret:
            break
            
        if i % 5 == 0:  # Check every 5th frame for speed
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            
            if prev_frame is not None:
                diff = cv2.absdiff(prev_frame, gray)
                mean_diff = np.mean(diff)
                
                if mean_diff > threshold:
                    scene_cuts.append(i)
                    
            prev_frame = gray
    
    cap.release()
    scene_cuts.append(frame_count - 1)
    
    return scene_cuts, fps

def select_best_scene(scene_cuts, fps, min_sec, max_sec):
    """Select best scene based on duration"""
    best_scene = None
    best_duration = 0
    
    for i in range(len(scene_cuts) - 1):
        start = scene_cuts[i]
        end = scene_cuts[i + 1]
        duration_sec = (end - start) / fps
        
        if min_sec <= duration_sec <= max_sec:
            if duration_sec > best_duration:
                best_duration = duration_sec
                best_scene = (start, end)
    
    return best_scene

# Analyze video
scene_cuts, fps = find_scene_cuts(video_file)
best_scene = select_best_scene(scene_cuts, fps, MIN_SHOT_SEC, MAX_SHOT_SEC)

if best_scene:
    begin_frame, end_frame = best_scene
    duration = (end_frame - begin_frame) / fps
    
    scene_info = {
        "best_shot_start_frame": int(begin_frame),
        "best_shot_end_frame": int(end_frame),
        "fps": fps,
        "duration_seconds": duration
    }
    
    with open("scene_analysis.json", "w") as f:
        json.dump(scene_info, f, indent=2)
    
    print(f"✅ Best scene found!")
    print(f"   Frames: {begin_frame} - {end_frame}")
    print(f"   Duration: {duration:.2f} seconds")
    print(f"   FPS: {fps}")
else:
    print("❌ No suitable scene found. Try adjusting MIN_SHOT_SEC and MAX_SHOT_SEC.")
    raise Exception("No suitable scene found")

---
## 🖼️ Step 6: Extract First Frame

In [ ]:
print("🖼️ Extracting first frame...\n")

video = cv2.VideoCapture(video_file)
video.set(cv2.CAP_PROP_POS_FRAMES, begin_frame)
success, frame = video.read()

if success:
    cv2.imwrite("first_frame.png", frame)
    print(f"✅ Frame {begin_frame} extracted and saved")
    print(f"   Size: {frame.shape[1]}x{frame.shape[0]}")
    
    # Display
    from IPython.display import Image, display
    display(Image('first_frame.png', width=600))
else:
    print(f"❌ Failed to extract frame {begin_frame}")
    raise Exception("Frame extraction failed")

video.release()

---
## 📐 Step 7: Plan Product Placement

In [ ]:
print("📐 Planning product placement...\n")

# Simple placement planner (center-bottom placement)
# In production, you'd use Nova API for intelligent placement

from PIL import Image

frame_img = Image.open('first_frame.png')
width, height = frame_img.size

# Place in bottom-right area (typical for product placement)
product_width = int(width * 0.15)   # 15% of frame width
product_height = int(height * 0.20)  # 20% of frame height

x1 = int(width * 0.70)  # 70% from left
y1 = int(height * 0.65)  # 65% from top
x2 = x1 + product_width
y2 = y1 + product_height

bbox = (x1, y1, x2, y2)

# Save bbox
bbox_dict = {
    "x1": x1,
    "y1": y1,
    "x2": x2,
    "y2": y2
}

with open("bbox.json", "w") as f:
    json.dump(bbox_dict, f, indent=2)

print(f"✅ Placement planned!")
print(f"   Bounding box: {bbox}")
print(f"   Product size: {product_width}x{product_height}")

# Visualize placement
import cv2
preview = cv2.imread('first_frame.png')
cv2.rectangle(preview, (x1, y1), (x2, y2), (0, 255, 0), 3)
cv2.imwrite('placement_preview.png', preview)

print("\n📦 Preview of placement area:")
display(Image('placement_preview.png', width=600))

---
## 🎨 Step 8: Load FLUX Model

In [ ]:
import torch
import os
from pathlib import Path
from PIL import Image
from finegrain_toolbox.src.finegrain_toolbox.flux import Model, TextEncoder
from finegrain_toolbox.src.finegrain_toolbox.processors import product_placement
from huggingface_hub import hf_hub_download
from optimum.quanto import quantize, freeze, qfloat8

print("="*60)
print("LOADING FLUX MODELS")
print("="*60)

device = torch.device("cuda")
dtype = torch.bfloat16

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

print("\n📥 Loading FLUX Kontext model...")
model = Model.from_pretrained(
    "black-forest-labs/FLUX.1-Kontext-dev",
    device=device,
    dtype=dtype,
)
print("✅ Model loaded")

print("\n📥 Loading text encoder...")
text_encoder = TextEncoder.from_pretrained(
    "black-forest-labs/FLUX.1-Kontext-dev",
    device=device,
    dtype=dtype,
)
print("✅ Text encoder loaded")

print("\n📥 Loading LoRA adapter...")
lora_path = Path(
    hf_hub_download(
        repo_id="finegrain/finegrain-product-placement-lora",
        filename="finegrain-placement-v1-rank8.safetensors",
    )
)
model.transformer.load_lora_adapter(lora_path, adapter_name="inserter")
print("✅ LoRA adapter loaded")

if USE_QUANTIZATION:
    print("\n⚙️ Quantizing model...")
    quantize(model.transformer, weights=qfloat8)
    freeze(model.transformer)
    print("✅ Model quantized")

print("\n" + "="*60)
print("🎉 ALL MODELS READY!")
print("="*60)
print(f"💾 Memory Used: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

---
## 🎨 Step 9: Generate Product Placement

In [ ]:
print("="*60)
print(f"GENERATING PRODUCT PLACEMENT: {PRODUCT_DESCRIPTION}")
print("="*60)

# Load images
scene_image = Image.open('first_frame.png')
reference_image = Image.open(product_file)

print(f"\n📂 Scene: {scene_image.size}")
print(f"📂 Product: {reference_image.size}")
print(f"📍 Bbox: {bbox}")

# Create prompt
prompt_text = f"Add the reference image of the product {PRODUCT_DESCRIPTION} in the box to look natural."
print(f"\n💬 Prompt: {prompt_text}")

prompt = text_encoder.encode(prompt_text)

# Clear cache
torch.cuda.empty_cache()

print("\n🎨 Generating... (this takes 2-3 minutes)")
print("⏱️  Please wait...\n")

result = product_placement.process(
    model=model,
    scene=scene_image,
    reference=reference_image,
    bbox=bbox,
    prompt=prompt,
)

result.output.save("flux_output.png")

print("\n✅ Product placement generated!")
print(f"💾 Saved as: flux_output.png")
print(f"💾 GPU Memory Used: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

print("\n📸 Result:")
display(Image('flux_output.png', width=800))

---
## 🎥 Step 10: Generate Full Video

In [ ]:
print("🎥 Generating full video with product placement...\n")

import cv2
import numpy as np
from ultralytics import YOLO

def generate_video_with_product(
    composited_frame_path,
    video_path,
    begin_frame,
    end_frame,
    product_bbox,
    output_path="final_video.mp4"
):
    """Generate video with product placement"""
    
    x1, y1, x2, y2 = map(int, product_bbox)
    
    # Load composited frame
    composited_img = cv2.imread(composited_frame_path)
    product_patch = composited_img[y1:y2, x1:x2].copy()
    
    # Open video
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    # Video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    cap.set(cv2.CAP_PROP_POS_FRAMES, begin_frame)
    
    print(f"Processing frames {begin_frame} to {end_frame}...")
    
    for frame_idx in range(begin_frame, end_frame + 1):
        ret, frame = cap.read()
        if not ret:
            break
        
        # Paste product patch
        frame[y1:y2, x1:x2] = product_patch
        
        out.write(frame)
        
        if (frame_idx - begin_frame) % 10 == 0:
            progress = ((frame_idx - begin_frame) / (end_frame - begin_frame)) * 100
            print(f"Progress: {progress:.1f}%", end='\r')
    
    cap.release()
    out.release()
    
    print(f"\n✅ Video generated: {output_path}")
    return output_path

# Generate video
final_video = generate_video_with_product(
    composited_frame_path='flux_output.png',
    video_path=video_file,
    begin_frame=begin_frame,
    end_frame=end_frame,
    product_bbox=bbox,
    output_path='final_video_with_product.mp4'
)

print(f"\n🎉 COMPLETE! Final video ready for download.")

---
## ⬇️ Step 11: Download Final Video

In [ ]:
from google.colab import files
import os

print("⬇️ Downloading final video...\n")

file_size_mb = os.path.getsize('final_video_with_product.mp4') / (1024 * 1024)
print(f"File: final_video_with_product.mp4")
print(f"Size: {file_size_mb:.2f} MB")
print(f"\nDownload will start shortly...")

files.download('final_video_with_product.mp4')

print("\n✅ Download complete!")
print("\n📋 Next Steps:")
print("1. Move video to: frontend/prime-video-ui/public/videos/")
print("2. Start frontend: npm run dev")
print("3. Open: http://localhost:5173")
print("4. Watch your video with product ads!")

---
## 📊 Summary

In [ ]:
print("="*60)
print("PROCESSING SUMMARY")
print("="*60)
print(f"\n✅ Input Video: {video_file}")
print(f"✅ Product: {PRODUCT_DESCRIPTION}")
print(f"✅ Scene: Frames {begin_frame}-{end_frame}")
print(f"✅ Duration: {(end_frame - begin_frame) / fps:.2f} seconds")
print(f"✅ Output: final_video_with_product.mp4")
print(f"\n🎉 Processing complete!")
print("="*60)